# L8

Na tych ?wiczeniach om?wimy:
* ensemble model?w,
* bagging,
* Random Forest,
* boosting (pokr?tce).

Bardzo dobrym uzupe?nieniem wiedzy z wyk?adu jest dokumentacja scikit-learn'a (om?wiono tam zalety i wady tych metod, ponadto podano wiele przyk?ad?w i praktycznych wskaz?wek):
* drzewa decyzyjne - http://scikit-learn.org/stable/modules/tree.html
* metody ensemble - http://scikit-learn.org/stable/modules/ensemble.html

Wikipedia te? daje rad?:
* https://en.wikipedia.org/wiki/Gradient_boosting
* https://en.wikipedia.org/wiki/Xgboost

## Ensemble

<img width=600 src="figures/L8/ensemble.jpg">

Je?li nauczyli?my na tych samych danych kilka silnych model?w, to zazwyczaj warto u?redni? ich przewidywania. W zale?no?ci od problemu musimy zawsze zastanowi? si?, co to znaczy "u?redni?":
* klasyfikacja - g?osowanie na klas? lub ?rednia arytmetyczna prawdopodobie?st predykcji poszczeg?lnych klas,
* regresja MSE - ?rednia arytmetyczna,
* przewidywanie rankingu - ?rednia arytmetyczna miejsc w rankingach itp.

?redni? mo?emy liczy? z wagami i wtedy wag poszczeg?lnych model?w szukamy dowolnym algorytmem doboru hiperparametr?w. Zazwyczaj jednak nie jest to konieczne.

### Bagging

* bagging jest metod? redukcji wariancji, nale?y go stosowa? jedynie z silnymi modelami, kt?re lubi? overfittowa?,
* idea:
  * losujemy dane ze zbioru treningowego z powt?rzeniami - je?li losujemy N z N przyk?ad?w, to otrzymamy ok. 63% r??nych wierszy (patrz wz?r z ?wiczenia L2a/7), dzi?ki temu na niekt?rych przypadkach skupimy si? bardziej (b?d? si? powtarza?y), a inne pominiemy,
  * mo?emy te? wylosowa? cz??? cech/kolumn (bez powt?rze?),
  * losowanie wykonujemy wielokrotnie i uczymy seri? model?w, a na ko?cu u?redniamy ich predykcje,
* Leo Breiman, "Bagging Predictors": http://link.springer.com/article/10.1023%2FA%3A1018054314350
* "Improvement will occur for unstable procedures where a small change in learning set can result in large changes in
prediction. Instability was studied in Breiman [1994] where it was pointed out that neural nets, classification and
regression trees, and subset selection in linear regression were unstable, while k-nearest neighbor methods were stable.",
* nie wymaga modyfikacji podstawowego algorytmu ucz?cego,
* mo?na uczy? r?wnolegle,
* wymaga wi?cej zasob?w.

### Boosting
* boosting polega na nauczeniu s?abego modelu, a nast?pnie poprawieniu go przy u?yciu kolejnego s?abego modelu, kt?ry przewiduje r??nic? mi?dzy predykcj? na zbiorze treningowym i prawdziwymi etykietami,
* trzeba u?ywa? s?abych model?w, kt?re nie overfituj?, poniewa? boosting zach?ca do overfittingu,
* Philip M. Long, "Random Classification Noise Defeats All Convex Potential Boosters" http://www.cs.columbia.edu/~rocco/Public/mlj9.pdf
* "In 2008 Phillip Long (at Google) and Rocco A. Servedio (Columbia University) published a paper[17] at the 25th International Conference for Machine Learning suggesting that many of these algorithms are probably flawed. They conclude that "convex potential boosters cannot withstand random classification noise," thus making the applicability of such algorithms for real world, noisy data sets questionable. The paper shows that if any fraction of the training data is mis-labeled, the boosting algorithm tries extremely hard to correctly classify these training examples, and fails to produce a model with accuracy better than 1/2. This result does not apply to branching program based boosters but does apply to AdaBoost, LogitBoost, and others.",
* nie wymaga modyfikacji podstawowego algorytmu ucz?cego,
* trzeba uczy? sekwencyjnie, wymaga wi?cej zasob?w.

### Gradient boosting vs Gradient descent
* licz?c $-\frac{\partial L(y_{true}, y_{pred})}{\partial y_{pred}}$ uzyskamy kierunek, w kt?rym powinni?my troszk? poprawi? $y_{pred}$ (predykcj? modelu), aby zmniejszy? ca?kowity koszt,
* je?li model jest r??niczkowalny, to mo?emy "pu?ci?" gradient dalej i obliczy? zmian? parametr?w modelu, kt?ra spowoduje po??dan? zmian? $y_{pred}$ - tak uczymy np. sieci neuronowe, regresj? liniow? itp.
* je?li model jest nier??niczkowalny, to zazwyczaj nie umiemy go zmodyfikowa? tak, aby odpowiednio zmieni? $y_{pred}$ - gradient boosting polega na nauczeniu drugiego modelu, kt?ry przybli?y $-\frac{\partial L(y_{true}, y_{pred})}{\partial y_{pred}}$, a nast?pnie przemno?eniu go przez pewn? ma?? liczb? rzeczywist? (steruj?c? d?ugo?ci? kroku) i dodaniem do modelu wyj?ciowego; proces powtarzamy wielokrotnie,
* stosujemy te? troch? innych trick?w, ale idea jest podobna,
* uwaga - zazwyczaj nie uda si? dobrze przybli?y? $-\frac{\partial L(y_{true}, y_{pred})}{\partial y_{pred}}$ (w szczeg?lno?ci, ?e musimy u?ywa? s?abych model?w), natomiast w przypadku model?w r??niczkowalnych kierunek ten jest wyznaczony dok?adnie.

### xgboost
* http://xgboost.readthedocs.io/en/latest/python/python_intro.html
* ma du?o sensownych hiperparametr?w,
* jest lepszy od Random Forest?w, prawie zawsze wygrywa lub jest cz??ci? rozwi?zania w konkursach na Kaggle'u,
* najlepsze narz?dzie do prostych problem?w klasyfikacji/regresji.

In [ ]:
# importujemy potrzebne biblioteki
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_mldata
from sklearn.decomposition import PCA
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split 
from sklearn.pipeline import Pipeline

# Dataset "breast cancer", by? ju? w L7
data = pd.read_csv("data/L8/breast_cancer_transformer.csv")
data.drop("Unnamed: 32",axis=1,inplace=True)
data['diagnosis']=data['diagnosis'].map({'M':1,'B':0})
prediction_var = [
    u'radius_mean', u'texture_mean', u'perimeter_mean',
    u'area_mean', u'smoothness_mean', u'compactness_mean',
    u'concavity_mean', u'concave points_mean', u'symmetry_mean',
    u'fractal_dimension_mean', u'radius_se', u'texture_se', u'perimeter_se',
    u'area_se', u'smoothness_se', u'compactness_se', u'concavity_se',
    u'concave points_se', u'symmetry_se', u'fractal_dimension_se',
    u'radius_worst', u'texture_worst', u'perimeter_worst', u'area_worst',
    u'smoothness_worst', u'compactness_worst', u'concavity_worst',
    u'concave points_worst', u'symmetry_worst', u'fractal_dimension_worst'
]
train, test = train_test_split(data, test_size = 0.3)
bc_train_X = train[prediction_var].values
bc_train_y=train.diagnosis.values
bc_test_X= test[prediction_var].values
bc_test_y =test.diagnosis.values

# MNIST
mnist = fetch_mldata('MNIST original')
mnist_X = mnist["data"]
mnist_y = mnist["target"].astype(np.int64)

# Syntetyczny dataset "diag", 2 r?wnoliczne klasy, rysunek poni?ej
how_many = 500
rng = np.random.RandomState(43)
_X = rng.uniform(size=how_many).reshape(-1,1)
X1 = np.hstack((_X, _X))
X1 += np.array([-0.1, 0.1])
X1 += rng.normal(scale=0.05, size=X1.shape)
_X = rng.uniform(size=how_many).reshape(-1,1)
X2 = np.hstack((_X, _X))
X2 += np.array([0.1, -0.1])
X2 += rng.normal(scale=0.05, size=X1.shape)
diag_X = np.vstack((X1, X2))
diag_y = np.hstack((np.zeros(how_many), np.ones(how_many)))
plt.figure(figsize=(5,5))
plt.scatter(X1[:,0], X1[:,1], color="b")
plt.scatter(X2[:,0], X2[:,1], color="r")
plt.title('"diag" dataset')
plt.show()

W ?wiczeniach 1. i 2. nale?y po wytrenowaniu modelu pokolorowa? obszary decyzyjne. Mo?na to zrobi? na przyk?ad tak: http://scikit-learn.org/stable/auto_examples/tree/plot_iris.html


### Feature engineering

Drzewa decyzyjne zak?adaj?, ?e kolumny tabeli X s? na tyle "sensowne" - mocno skorelowane z etykietami (same z siebie lub pod warunkiem innych cech) - ?e do predykcji etykiety wystarczy por?wnywanie ich z pewnymi ustalonymi thresholdami (regresja liniowa, SVM itp. bior? kombinacje liniowe cech, sieci neuronowe ucz? si? cech wysokopoziomowych). 

Oczywi?cie nie zawsze jest to prawd?, dlatego warto spr?bowa? doda? r?cznie nowe kolumny, na przyk?ad:
* sum?/r??nic? kilku kolumn,
* iloczyn kilku kolumn,
* logarytm kolumn,
* wielomian zale?ny od kilku kolumn,
* je?li cechy opisuj? tr?jwymiarowy obiekt i mamy podane 3 wymiary, to mo?e warto doda? obj?to??, ?rednic? itp.

Jest to analogiczne do zmiany zmiennych w regresji liniowej.

Zilustrujemy ten problem na przyk?adzie datasetu "diag" - zobaczymy, ?e drzewa decyzyjne maj? problem z "uko?nymi" granicami decyzyjnymi.

## ?wiczenie 1 [2 pkt]

Wytrenowa?:
* pojedyncze drzewo decyzyjne,
* Random Forest,
* pojedyncze drzewo decyzyjne po zamianie drugiej kolumny X na r??nic? pierwszej i drugiej kolumny X

dla datasetu "diag" i zaznaczy? obszary decyzyjne (3 modele i 3 ilustracje).

Je?li cechy (kolumny X) nie s? "sensowne", to by? mo?e warto zastosowa? metody redukcji wymiarowo?ci typu PCA, ICA (czy by?y na Analizie Danych?). PCA spe?nia dwie funkcje:
* wyszukuje **kombinacje liniowe** cech, kt?re maj? du?? wariancj?, a wi?c potencjalnie mog? by? bardziej skorelowane z etykietami (przyk?ad: https://en.wikipedia.org/wiki/Eigenface) - to dzia?a nawet, gdy nie zmniejszamy liczby cech!
* redukcja wymiaru mo?e zmniejszy? overfitting.

Oczywi?cie mo?e si? te? zdarzy?, ?e PCA bardziej przeszkadza, ni? pomaga - u?ycie redukcji wymiarowo?ci danych nale?y zawsze traktowa? jako kolejny hiperparametr.

## ?wiczenie 2 [2 pkt]

U?ywaj?c klasy Pipeline po??czy? PCA (bez zmniejszania wymiaru) z pojedynczym drzewem decyzyjnym i wytrenowa? na datasecie "diag". Zaznaczy? kolorem obszary decyzyjne. Wyt?umaczy? wynik.

### Random Forest

Dodanie wi?kszej liczby drzew powoduje zwi?kszenie liczby parametr?w modelu. Mog?oby si? wydawa?, ?e spowoduje to w ko?cu overfitting i spadek skuteczno?ci na zbiorze testowym, ale okazuje si?, ?e Random Forest jest na to stosunkowo odporny.

Z regu?y im wi?cej u?yjemy drzew, tym lepiej dla predykcji, ale tym gorzej dla naszych zasob?w (czas uczenia, RAM, storage nauczonego modelu itp.).

Jak dobiera? parametry: http://scikit-learn.org/stable/modules/ensemble.html#parameters

Parallelization: http://scikit-learn.org/stable/modules/ensemble.html#parallelization

Prosz? w szczeg?lno?ci zwr?ci? uwag? na parametr "oob_score".

## ?wiczenie 3 [2 pkt]

Na datasecie "breast cancer" nale?y wytrenowa? dwie serie model?w: Random Forest oraz ensemble regresji logistycznych (BaggingClassifier w po??czeniu z LogisticRegression). Nale?y w??czy? liczenie oob_score, mo?na te? uczy? wielow?tkowo. Nale?y zostawi? domy?lne warto?ci wszystkich hiperparametr?w poza liczb? estymator?w.

Narysowa? wykres:
* poziomo - liczba estymator?w (w zakresie np. 1-50, mo?na przeskakiwa? co kilka),
* pionowo sze?? rzeczy:
    * Random Forest training score,
    * Random Forest test score,
    * Random Forest oob_score,
    * Ensemble LR training score,
    * Ensemble LR test score,
    * Ensemble LR oob_score,

doda? ?adn? legend? itp.

### Feature selection

Random Forest potrafi po wytrenowaniu oceni?, kt?re cechy ze zbioru treningowego by?y najistotniejsze dla poprawnej predykcji. Wiedz? t? mo?emy wykorzysta? na wiele sposob?w:
* je?li pr?bujemy zinterpretowa? dane i model, mo?emy skupi? si? tylko na najwa?niejszych cechach,
* je?li projektujemy maszyn? i chcemy do??czy? do niej system wczesnego ostrzegania przed awari?, to mo?emy na etapie projektowania zamontowa? bardzo du?o kosztownych czujnik?w, wytrenowa? Random Forest (przewidywanie awarii z odczyt?w z czujnik?w) i w produkcji masowej zostawi? tylko te najistotniejsze sensory,
* czasem mo?emy poprawi? wynik Random Forest'u dzia?aj?c w nast?puj?cy spos?b: trenujemy pierwszy raz, usuwamy wszystkie nieistotne cechy (musimy r?cznie dobra? poziom istotno?ci), trenujemy drugi raz tylko na istotnych cechach,
* itp.

## ?wiczenie 4 [2 pkt]

Wytrenowa? Random Forest (30 lub 100 drzew) na datasecie MNIST. Narysowa? istotno?? poszczeg?lnych pikseli. Ograniczy? si? tylko do cyfr 6, 8, 9 i zrobi? to samo, wyja?ni? r??nic?.

Przyk?ad dla zdj?? twarzy: http://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances_faces.html

Oczekiwany efekt:
<img width=300 src="figures/L8/mnist_features.png">
<img width=300 src="figures/L8/mnist_features_689.png">
